# 4. Modelos de Machine Learning

En este notebook se entrenan diferentes modelos de aprendizaje supervisado para predecir la supervivencia en el Titanic.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None

In [ ]:
X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")
y_train = pd.read_csv("data/y_train.csv").values.ravel()
y_test = pd.read_csv("data/y_test.csv").values.ravel()

X_train.head()

## Diccionario de modelos

Cada modelo se guarda junto con los hiperparámetros que se probarán con GridSearchCV.

In [ ]:
modelos = {
    "Regresión Logística": {
        "modelo": LogisticRegression(),
        "parametros": {
            "C": [0.01, 0.1, 1, 10],
            "penalty": ["l1", "l2"],
            "solver": ["liblinear"],
            "max_iter": [500, 1000]
        }
    },
    "Clasificador de Vectores de Soporte": {
        "modelo": SVC(),
        "parametros": {
            "kernel": ["linear", "rbf"],
            "C": [0.1, 1, 10]
        }
    },
    "Clasificador de Árbol de Decisión": {
        "modelo": DecisionTreeClassifier(random_state=42),
        "parametros": {
            "splitter": ["best", "random"],
            "max_depth": [None, 2, 3, 4]
        }
    },
    "Clasificador de Bosques Aleatorios": {
        "modelo": RandomForestClassifier(random_state=42),
        "parametros": {
            "n_estimators": [10, 100],
            "max_depth": [None, 2, 3, 4],
            "max_features": ["sqrt", "log2", None]
        }
    },
    "Clasificador de Gradient Boosting": {
        "modelo": GradientBoostingClassifier(random_state=42),
        "parametros": {
            "n_estimators": [10, 100],
            "max_depth": [1, 2, 3]
        }
    },
    "Clasificador AdaBoost": {
        "modelo": AdaBoostClassifier(random_state=42),
        "parametros": {
            "n_estimators": [10, 100]
        }
    },
    "Clasificador K-Nearest Neighbors": {
        "modelo": KNeighborsClassifier(),
        "parametros": {
            "n_neighbors": [3, 5, 7]
        }
    },
    "GaussianNB": {
        "modelo": GaussianNB(),
        "parametros": {}
    },
    "Clasificador Naive Bayes": {
        "modelo": BernoulliNB(),
        "parametros": {
            "alpha": [0.1, 1.0, 10.0]
        }
    }
}

if XGBClassifier is not None:
    modelos["Clasificador XGBoost"] = {
        "modelo": XGBClassifier(eval_metric="logloss", random_state=42),
        "parametros": {
            "n_estimators": [10, 100],
            "max_depth": [1, 2, 3]
        }
    }

if LGBMClassifier is not None:
    modelos["Clasificador LGBM"] = {
        "modelo": LGBMClassifier(verbose=-1, random_state=42),
        "parametros": {
            "n_estimators": [10, 100],
            "max_depth": [1, 2, 3],
            "learning_rate": [0.1, 0.2, 0.3]
        }
    }

## Entrenamiento con GridSearchCV

GridSearchCV prueba diferentes combinaciones de hiperparámetros y selecciona la mejor opción según la métrica accuracy.

In [ ]:
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}

for nombre, info_modelo in modelos.items():
    grid_search = GridSearchCV(
        estimator=info_modelo["modelo"],
        param_grid=info_modelo["parametros"],
        cv=5,
        scoring="accuracy",
        verbose=0,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)
    y_pred = grid_search.predict(X_test)
    precision = accuracy_score(y_test, y_pred)

    puntajes_modelos.append({
        "Modelo": nombre,
        "Precisión": precision
    })

    estimadores[nombre] = grid_search.best_estimator_

    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

metricas = pd.DataFrame(puntajes_modelos).sort_values("Precisión", ascending=False)

print("Rendimiento de los modelos de clasificación")
display(metricas.round(2))

print("---------------------------------------------------")
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.2f}")

## Conclusión

El mejor modelo se selecciona con base en su precisión sobre los datos de prueba. Esto permite comparar modelos de forma objetiva y no solo por intuición.